In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import config
import json

from config import DATA_DIR, DIMENSIONS

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from Analysis.mfrm_analysis import read_filepaths

raw_old_data_dir = Path("../hate-thermometer-anonymous/hate_thermometer_data/data/raw_data")
repatched_old_data_dir = Path("../hate-thermometer-anonymous/hate_thermometer_data/data/patched_data")
refusal_dir = Path("../hate-thermometer-anonymous/hate_thermometer_data/data/refusal_analysis")
refusal_ord_dir = Path("../hate-thermometer-anonymous/hate_thermometer_data/data/refusal_analysis/refusal_ordinal_only")

refusal_dir.mkdir(parents=True, exist_ok=True)
refusal_ord_dir.mkdir(parents=True, exist_ok=True)

refusal_table = {}
cols = ['text', 'comment_id', 'annotator_id', 'vote_distributions'] + DIMENSIONS
NON_ORD_DIMS = DIMENSIONS[1:-1]

class RefusalAnalysis:
    def load_data(file_path):
        try:
            data = pd.read_csv(file_path)
            return data
        except Exception as e:
            print(f"Error loading data: {e}")
            return None

    def count_valid_runs(vote_dist, dim):
        if dim not in vote_dist:
            return 0
        counts = vote_dist[dim]
        return sum(counts.values())

    def count_incompletes_per_file(file_directory):
        for data_name, file_path in file_directory.items():
            data = load_data(file_path)
            if 'vote_distributions' not in data.columns:
                print(f"Missing vote_distributions: {file_path}")
                continue
            vote_dist = data['vote_distributions'].apply(
                lambda x: json.loads(x) 
            if isinstance(x, str) else x)

            incompletes = {}
            for dim in DIMENSIONS:
                valid_runs   = data['vote_distributions'].apply(
                    lambda x: count_valid_runs(json.loads(x), dim))

                # Count valid runs
                data[f'{dim}_runs'] = vote_dist.apply(
                    lambda vd: sum(vd.get(dim, {}).values()) )
                
                data[f'{dim}_incomplete'] = valid_runs < 5
                count = data[f'{dim}_incomplete'].sum()
                pct = count/len(data)*100
                print(f"{data_name:20s} {dim:15s}: {count:4d} ({pct:5.1f}%)")
                print()
                incompletes[dim] = count
            refusal_table[data_name] = incompletes
            data['any_incomplete'] = data[
                [f'{dim}_incomplete' for dim in DIMENSIONS]
                ].any(axis=1)

            data_all_ord_empty = data[
            [f'{dim}_runs' for dim in ORD_DIMENSIONS]
                ].eq(0).all(axis=1)

            data['any_ord_incomplete'] = data[
                [f'{dim}_runs' for dim in ORD_DIMENSIONS]].apply(
                lambda row: any(0 < v < 5 for v in row),axis=1)

            incomplete_output_file = refusal_dir / f"{data_name}_incompletes.csv"
            ord_output_file = refusal_ord_dir / f"{data_name}_ord_incompletes.csv"
            # (data.loc[data['any_incomplete'], cols].to_csv(incomplete_output_file, index=False))
            # (data.loc[data['any_ord_incomplete'], cols].to_csv(ord_output_file, index=False))
            print(f"  Total:          {len(data)}, model: {data_name}") 
            print(f"  Any incomplete: {data['any_incomplete'].sum()}")
            print(f"  Pipeline branch:{data_all_ord_empty.sum()}")
            print(f"Any ordinal incomplete: {data['any_ord_incomplete'].sum()}")
            print(f"percentage of any incomplete: {data['any_incomplete'].sum()/len(data)*100}%")
            print(f"percentage of any ordinal incomplete(acrodd all data): {data['any_ord_incomplete'].sum()/len(data)*100}%")
            print(f"percentage of any ordinal incomplete (acoss incompletes only): {data['any_ord_incomplete'].sum()/data['any_incomplete'].sum()*100}%")
        summary = pd.DataFrame(refusal_table).T
        # summary.to_csv(refusal_dir / "refusal_summary.csv")
        return summary


# Incomplete runs bafore repatch

In [ ]:
file_directory = read_filepaths(raw_old_data_dir)
summary = RefusalAnalysis.count_incompletes_per_file(file_directory)
print("Refusal analysis completed. Summary:")
print(summary)

# Incomplete runs after repatch

In [ ]:
patched_file_directory = read_filepaths(repatched_old_data_dir)
summary_patched = RefusalAnalysis.count_incompletes_per_file(patched_file_directory)
print("Refusal analysis completed. Summary:")
print(summary_patched)

In [ ]:
# import wordcloud
from collections import Counter
import re
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def tokenizeandstopwords(text):
    tokens = nltk.word_tokenize(text)
    # taken only words (not punctuation)
    token_words = [w for w in tokens if w.isalpha()]
    meaningful_words = [w for w in token_words if not w in stop_words]
    joined_words = ( " ".join(meaningful_words))
    return joined_words

def word_count_analysis(incomplete_ord_df, model_name, title=""):
    all_text = ' '.join(incomplete_ord_df['text'])
    print(all_text[:500])  # Print the first 500 characters of the combined text    
    
    # # Tokenize and clean
    # words = re.findall(r'\b[a-zA-Z]{3,}\b', 
    #                 all_text.lower())

    words = tokenizeandstopwords(all_text.lower()).split()
    print(f"Total words after tokenization and stopword removal: {len(words)}")
    
    # # Remove common stopwords
    # words = [w for w in words if w not in stop_words]
    
    # Count
    word_counts = Counter(words)
    top_words = word_counts.most_common(10)
    
    # Plot
    words_list, counts = zip(*top_words)
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(words_list[::-1], 
                counts[::-1], 
                color='steelblue',
                alpha=0.7)
    ax.set_xlabel('Frequency')
    ax.set_title(f'Top 10 Words in Incomplete Annotations {model_name}\n{title}')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(f'word_count_{title.replace(" ", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    return pd.DataFrame(top_words, columns=['word', 'count'])


all_filepaths = read_filepaths(DATA_DIR)

for model_name, filepath in all_filepaths.items():
    # model_name = filepath.stem 
    data = pd.read_csv(filepath)
    print(f"\n=== ordinal Runs only - no demographic targetting for {model_name} ===")
    type2_words = word_count_analysis(
        data,
        model_name,
        title="Partial Runs (< 5 valid runs) ordinal"
    )
    print(f"\nTop 10 words for {model_name}:")
    print(type2_words.head(10))

In [ ]:
patched_file_directory = read_filepaths(repatched_old_data_dir)
summary_patched = RefusalAnalysis.count_incompletes_per_file(patched_file_directory)
print("Refusal analysis completed. Summary:")
print(summary_patched)

# Only the incomplate runs

In [ ]:
from Analysis.mfrm_analysis import read_filepaths

DATA_DIR = Path(".../data_outputs_stats/data/refusal_analysis/refusal_ordinal_only")

all_filepaths = read_filepaths(DATA_DIR)
# filepath_gold = all_filepaths['gold']
filepath_GPT5= all_filepaths['GPT5_mono']
filepath_claude_modu= all_filepaths['claude_modu']
filepath_claude_mono= all_filepaths['claude_mono']
filepath_gemini_modu= all_filepaths['gemini_modu']
filepath_gemini_mono= all_filepaths['gemini_mono']
filepath_deepseek_modu= all_filepaths['deepseek_modu']
filepath_deepseek_mono= all_filepaths['deepseek_mono']
filepath_gpt5mini_modu= all_filepaths['gpt_5-mini_modu']
filepath_gpt5mini_mono= all_filepaths['gpt_5-mini_mono']

In [ ]:
ORD_DIMENSIONS = DIMENSIONS[1:-1]
from Analysis.mfrm_analysis import read_filepaths

def ordinal_incompletes(refusal_file_directory):
    for data_name, file_path in file_directory.items():
        data = load_data(file_path)
        vote_dist = data['vote_distributions'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x)
        # ordinal dimensions with < 5 runs
        data['any_ord_incomplete'] = data[
            [dim for dim in ORD_DIMENSIONS]
        ].apply(lambda row: any(0 < v < 5 for v in row), axis=1)
        # Partial run cases
        ord_incompletes = data.loc[data['any_ord_incomplete'], cols].copy()
        ord_incompletes.to_csv( refusal_ord_dir/ f"{data_name}_ord_incompletes.csv", index=False)
    
    return ord_incompletes


# ── RUN ──────────────────────────────────────────────────────
# data = pd.read_csv(filepath_claude_mono)
refusal_file_directory = read_filepaths(refusal_dir)

type2_df = ordinal_incompletes(refusal_file_directory)
type2_df.head()